# Benchmark Karsilastirmasi
Portfolyo olmadan: Altin, Gumus, Dolar, Euro, BIST100 ve Mevduat faizi karsilastirmasi.
Tum varliklar baslangic=100 bazinda normalize edilir.

In [1]:
# Hucre 1 - Bagimlilik kurulumu
import subprocess, sys

REQUIRED = ["yfinance", "plotly", "ipywidgets"]
for pkg in REQUIRED:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Bagimliliklar hazir")

Bagimliliklar hazir


In [2]:
# Hucre 2 - Importlar + Google Drive baglama
import os, sys
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive, output
    drive.mount('/content/drive')
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
    print("Google Drive baglandi | Colab widget/Plotly renderer hazir")
else:
    print("Yerel ortam - Drive mount atlandi")

Yerel ortam - Drive mount atlandi


In [3]:
# Hucre 3 - Konfigurasyon
PROJECT_ROOT_CANDIDATES = [
    os.getcwd(),
    "/content/BenchmarkTakip",
    "/content/drive/MyDrive/PortfolioProject",
]
PROJECT_ROOT = next(
    (p for p in PROJECT_ROOT_CANDIDATES if os.path.isdir(os.path.join(p, "lib"))),
    os.getcwd(),
)

DRIVE_DATA_DIR = "/content/drive/MyDrive/PortfolioProject"
DRIVE_BASE = os.environ.get("PORTFOLIO_DATA_DIR")
if not DRIVE_BASE:
    if IN_COLAB and os.path.isdir(DRIVE_DATA_DIR):
        DRIVE_BASE = DRIVE_DATA_DIR
    else:
        DRIVE_BASE = os.path.join(PROJECT_ROOT, "data")
DRIVE_BASE = os.path.abspath(DRIVE_BASE) + os.sep

CACHE_PATH = os.path.join(DRIVE_BASE, "cache")

# Karsilastirilacak benchmark'lar
SYMBOLS = {
    "Gram Altin": "GC=F",
    "Gram Gumus": "SI=F",
    "DOLAR":      "USDTRY=X",
    "EURO":       "EURTRY=X",
    "BIST100":    "XU100.IS",
}

TCMB_POLICY_RATE_PCT = 37  # Guncelle: mevcut TCMB politika faizi

# Varsayilan baslangic tarihi
DEFAULT_START = "2023-01-01"
DEFAULT_END   = datetime.today().strftime("%Y-%m-%d")

print(f"Config: {len(SYMBOLS)} benchmark sembol | PROJECT_ROOT={PROJECT_ROOT} | DATA={DRIVE_BASE}")

Config: 5 benchmark sembol


In [4]:
# Hucre 4 - lib/ import
LIB_PATH = os.path.join(PROJECT_ROOT, "lib")
if not os.path.isdir(LIB_PATH):
    raise FileNotFoundError(
        f"lib klasoru bulunamadi: {LIB_PATH}. Colab'da once repoyu clone edip os.chdir(repo_klasoru) yapin."
    )

if LIB_PATH not in sys.path:
    sys.path.insert(0, LIB_PATH)

from data_loader import fetch_prices, load_cpi_series, load_tcmb_rates
from benchmark_engine import build_benchmark_series, build_deposit_series
from chart_builder import build_performance_line_chart
from widgets import create_date_range_picker, create_currency_toggle, wire_dashboard

print("lib/ moduller yuklendi")

from chart_builder_v2 import (
    build_performance_line_chart_v2,
    build_asset_filter_widget,
    build_drawdown_chart,
    build_correlation_heatmap,
    build_period_bar_chart,
    build_treemap,
    build_risk_return_scatter,
)


lib/ moduller yuklendi


In [5]:
# Hucre 5 - Veri yukle
os.makedirs(CACHE_PATH, exist_ok=True)

CPI_PATH  = os.path.join(DRIVE_BASE, "cpi_turkey.csv")
TCMB_PATH = os.path.join(DRIVE_BASE, "tcmb_rates.csv")

cpi_series  = load_cpi_series(CPI_PATH)
tcmb_rates  = load_tcmb_rates(TCMB_PATH, policy_rate_pct=TCMB_POLICY_RATE_PCT)

all_symbols = list(SYMBOLS.values()) + ["USDTRY=X"]
prices = fetch_prices(all_symbols, start=DEFAULT_START, end=DEFAULT_END, cache_path=CACHE_PATH)
fx_usdtry = prices["USDTRY=X"].dropna()

print(f"Veri hazir | Aralik: {DEFAULT_START} - {DEFAULT_END}")

Veri hazir | Aralik: 2023-01-01 - 2026-05-13


In [6]:
# Hucre 6 - Dashboard
output = widgets.Output()

CURRENCY_LABELS = {"TL": "TL (Nominal)", "USD": "USD", "REAL": "Reel (TUFE)"}
sym_to_name = {v: k for k, v in SYMBOLS.items()}

def render(start_date, end_date, currency):
    benchmark_df = build_benchmark_series(
        symbols=list(SYMBOLS.values()),
        start_date=start_date,
        end_date=end_date,
        prices=prices,
        fx_usdtry=fx_usdtry,
        cpi_series=cpi_series if currency == "REAL" else None,
        currency=currency,
    )
    deposit_series = build_deposit_series(tcmb_rates, start_date, end_date)
    benchmark_df["Mevduat"] = deposit_series

    benchmark_df = benchmark_df.rename(columns=sym_to_name)

    currency_label = CURRENCY_LABELS.get(currency, currency)

    line_fig = build_performance_line_chart(
        portfolio_series=None,
        benchmark_series=benchmark_df,
        currency_label=currency_label,
        title=f"Benchmark Karsilastirmasi ({currency_label})",
    )
    display(line_fig)

    son_degerler = benchmark_df.iloc[-1].dropna()
    getiri_df = pd.DataFrame({
        "Varlik": son_degerler.index,
        "Son Deger (baz=100)": son_degerler.values.round(2),
        "Toplam Getiri %": (son_degerler.values - 100).round(2),
    }).sort_values("Toplam Getiri %", ascending=False)

    tbl = go.Figure(go.Table(
        header=dict(
            values=["<b>Varlik</b>", "<b>Son Deger</b>", "<b>Getiri %</b>"],
            fill_color="#313244",
            font=dict(color="#cdd6f4", size=12),
            align="center",
        ),
        cells=dict(
            values=[
                getiri_df["Varlik"],
                getiri_df["Son Deger (baz=100)"],
                [f"{v:+.2f}%" for v in getiri_df["Toplam Getiri %"]],
            ],
            fill_color=[["#1e1e2e" if i % 2 == 0 else "#181825" for i in range(len(getiri_df))]],
            font=dict(color="#cdd6f4", size=11),
            align="center",
        ),
    ))
    tbl.update_layout(
        title="Donem Sonu Getiri Ozeti",
        template="plotly_dark",
        height=max(200, 38 * len(getiri_df) + 60),
        margin=dict(l=0, r=0, t=40, b=0),
    )
    display(tbl)

start_picker, end_picker = create_date_range_picker(
    min_date=datetime(2015, 1, 1),
    max_date=datetime.today(),
    default_start=datetime.strptime(DEFAULT_START, "%Y-%m-%d"),
    default_end=datetime.today(),
)
currency_toggle = create_currency_toggle()

dashboard = wire_dashboard(
    render_fn=render,
    output_widget=output,
    start_picker=start_picker,
    end_picker=end_picker,
    currency_toggle=currency_toggle,
)

with output:
    render(DEFAULT_START, DEFAULT_END, "TL")

display(dashboard)

---
## Detaylı Analiz Grafikleri

Aşağıdaki hücreler interaktif analiz grafikleri içerir. Her grafik öncesinde açıklama bulunur.
Grafikleri çalıştırmak için önce **Hücre 1-5** (kurulum ve veri yükleme) tamamlanmış olmalıdır.
Tüm grafikler **Başlangıç = 100** bazında normalize edilmiş veriyi kullanır.

In [7]:
# Analiz bölümü için paylaşılan veri seti
# Bu hücreyi çalıştırın; aşağıdaki tüm grafik hücreleri bu DataFrame'i kullanır
analysis_df = build_benchmark_series(
    symbols=list(SYMBOLS.values()),
    start_date=DEFAULT_START,
    end_date=DEFAULT_END,
    prices=prices,
    fx_usdtry=fx_usdtry,
    cpi_series=cpi_series,
    currency="TL",
)
analysis_deposit = build_deposit_series(tcmb_rates, DEFAULT_START, DEFAULT_END)
analysis_df["Mevduat"] = analysis_deposit.reindex(analysis_df.index).ffill().bfill()
analysis_df = analysis_df.rename(columns=sym_to_name)

print(f"Analiz verisi hazir: {len(analysis_df)} gun, {list(analysis_df.columns)}")

Analiz verisi hazir: 878 gun, ['Gram Altin', 'Gram Gumus', 'DOLAR', 'EURO', 'BIST100', 'Mevduat']


### Performans Karşılaştırması (İnteraktif Filtre)

**Ne gösterir:** Tüm varlıkların başlangıç tarihinden bu yana normalize edilmiş getiri seyrini kıyaslar.
Tüm çizgiler **başlangıç = 100** baz alınarak yeniden ölçeklendirilmiştir; böylece farklı birimler
(TL/gram, dolar, puan) doğrudan karşılaştırılabilir. Kesikli çizgi kullanılmaz — tüm seriler solid.

**Nasıl kullanılır:**
- **Combobox:** Dropdown menüsünden tek bir varlık seçerek yalnızca o varlığın performansını inceleyin.
- **Range Selector:** Grafiğin üstündeki butonlarla (1A · 3A · 6A · YBB · 1Y · Tümü) zaman aralığını daraltın.
- **Range Slider:** Grafik altındaki mini çubuğu sürükleyerek özel bir zaman penceresi seçin.
- **Hover:** İmleç grafik üzerindeyken tarih, güncel değer ve başlangıçtan % değişimi okunur.
- **Legend:** Bir varlık adına tıklayarak o çizgiyi gizleyin; çift tıkla ile sadece onu görüntüleyin.

In [ ]:
# İnteraktif performans grafiği — dropdown'dan varlık seçimi
# 'Tumü' = tüm varlıklar; tek isim seçilince sadece o varlık görünür
filter_widget = build_asset_filter_widget(
    benchmark_df=analysis_df,
    portfolio_series=None,
    currency_label="TL (Nominal)",
)
display(filter_widget)

### Maksimum Drawdown — Tepe'den Düşüş

**Ne gösterir:** Her varlığın kendi tarihsel tepe noktasından ne kadar geride kaldığını yüzde olarak gösterir.
Sıfır çizgisi o varlığın bir önceki tarihsel zirvesini temsil eder; aşağı yönlü bölgeler kayıp dönemlerini
işaret eder. Tepe yeniden kırıldığı anda değer sıfıra döner.

**Neden önemlidir:**
- Yatırımcının tahammül etmesi gereken maksimum geçici kaybı (max-drawdown) görselleştirir.
- İki varlık benzer toplam getiri sağlasa bile farklı risk profillerine sahip olabilir;
  daha düz seyreden varlık genellikle daha az psikolojik baskı yaratır.
- Portföy varsa kırmızı alan dolgusuyla vurgulanır.

In [9]:
# Drawdown grafiği
# Y ekseni: tepe noktasından yüzde düşüş (0 = tepede, negatif = kayıp bölgesi)
fig_dd = build_drawdown_chart(analysis_df)
fig_dd.show()

### Varlık Getiri Korelasyonu

**Ne gösterir:** Günlük yüzde getiriler arasındaki istatistiksel ilişkiyi ölçer.
Renk skalası: **kırmızı → −1** (tam zıt hareket) · **koyu zemin → 0** (ilişkisiz) · **yeşil → +1** (aynı yön).
Köşegen her zaman **1.00** olmalıdır (varlığın kendisiyle olan korelasyonu).

**Neden önemlidir:**
- Korelasyonu düşük veya negatif varlıklar bir arada tutulduğunda portföy volatilitesi azalır.
- Yüksek korelasyonlu iki varlığı birden tutmak gerçek anlamda çeşitlendirme sağlamaz.
- Örneğin Gram Altın ile Dolar/TL arasındaki korelasyon, TL değer kaybı dönemlerini anlamak
  için önemli bir sinyal olabilir.

In [10]:
# Korelasyon ısı haritası
# Günlük getiri değişimleri kullanılarak Pearson korelasyon matrisi hesaplanır
fig_corr = build_correlation_heatmap(analysis_df)
fig_corr.show()

### Dönemsel Getiri Karşılaştırması (Aylık / Çeyreklik)

**Ne gösterir:** Her dönemin (ay veya çeyrek) son günündeki kapanış değeri üzerinden hesaplanan
yüzde getiriyi gruplanmış çubuk grafik olarak sunar. Hover'da ilgili dönemin tam getirisi okunur.

**Nasıl okunur:**
- **Sıfır çizgisinin üstü:** o dönemde kazanç.
- **Sıfır çizgisinin altı:** o dönemde değer kaybı.
- Aynı dönemde farklı renkteki çubukları kıyaslayarak hangi varlığın öne çıktığını görün.
- Örneğin yüksek enflasyon dönemlerinde Altın çubuklarının diğerlerine göre konumuna bakın.

In [11]:
# Aylık dönemsel getiri karşılaştırması
fig_monthly = build_period_bar_chart(analysis_df, freq="ME")
fig_monthly.show()

# Çeyreklik dönemsel getiri karşılaştırması
fig_quarterly = build_period_bar_chart(analysis_df, freq="QE")
fig_quarterly.show()

### Katkı Haritası ve Risk-Getiri Dağılımı

**Katkı Haritası (Treemap):**
Her kutunun **büyüklüğü** varlığın mutlak toplam getirisini (P&L), **rengi** ise yüzde getirisini gösterir.
Yeşil = kazanç · Kırmızı = kayıp. Hover'da P&L tutarı, yüzde getiri, ağırlık ve portföy katkısı görünür.

**Risk-Getiri Dağılımı (Scatter):**
Her nokta bir varlığı temsil eder. **X ekseni** yıllık volatilite (risk ölçütü),
**Y ekseni** başlangıçtan bu yana toplam getiriyi gösterir.
**Hedef konum: sol üst** (yüksek getiri + düşük risk = verimli varlık).
Sağ alt köşedeki noktalar yüksek risk taşıyıp düşük getiri sunuyor demektir.
Portföy mevcutsa yıldız (★) sembolüyle gösterilir.

In [12]:
# Katkı haritası — toplam getiri bazlı (portföy verisi yoksa sentetik hesaplanır)
_last  = analysis_df.iloc[-1]
_first = analysis_df.iloc[0]
_ret   = (_last / _first - 1) * 100
_n     = len(analysis_df.columns)
_weight = 100.0 / _n

contributions_analysis = pd.DataFrame({
    "Varlik Adi":       analysis_df.columns.tolist(),
    "pnl_tl":           (_ret * 1000).tolist(),
    "pnl_pct":          _ret.tolist(),
    "weight_pct":       [_weight] * _n,
    "contribution_pct": (_weight / 100 * _ret).tolist(),
})
contributions_analysis = contributions_analysis.rename(columns={"Varlik Adi": "Varlık Adı"})

# Treemap: büyüklük = mutlak P&L, renk = yüzde getiri
fig_treemap = build_treemap(contributions_analysis)
fig_treemap.show()

# Risk-getiri scatter: her nokta bir varlık
fig_scatter = build_risk_return_scatter(analysis_df)
fig_scatter.show()